# 7 · Distributed SeQUeNCe BB84 over FABRIC

Run real **SeQUeNCe** `QKDNode` / BB84 instances on two FABRIC nodes that pass
**real traffic on the wire**: photons cross as `0x7101` Ethernet frames through the
BMv2 **P4 switch** (which applies fiber loss), while classical sifting and QBER
**sample disclosure** ride TCP over the data plane — the real-WAN lever.

This is the `qne-sequence/` runtime (see `qne-sequence/DESIGN.md`), the distributed
evolution of the hand-coded BB84 in notebook 2. The slice and data-plane IPs come from
notebook 1; only the **run** step differs (`qne_sequence.node_runner` instead of `qne.cli`).

**Channel modes** (set `TRANSPORT`/`LOSS` in cell 1) — runs **with or without** the P4 switch:
- `raw` + `switch` — real `0x7101` photons through the BMv2 **P4 switch** (loss in the data plane). *Default; needs notebook 1's switch.*
- `raw` + `model` — real `0x7101` photons node-to-node, **loss in software** (no switch).
- `tcp` — photon *descriptors* over TCP, software loss (**no switch, no root**).
- any transport + `none` — **lossless / ideal** channel (QBER from fidelity only).

### At a glance
- **Purpose:** run one distributed-SeQUeNCe BB84 experiment across the slice and record the result.
- **Prereqs:** **run notebook 1 first** — it provisions the slice, starts BMv2, and sets up the data-plane IPs/ARP/promisc. This notebook reuses that slice.
- **Inputs:** `SLICE_NAME`, `SCENARIO` (match notebook 1), and the emulator knobs below.
- **Outputs:** `results/fabric_seq_{alice,bob}.json`; on-node logs at `/tmp/seq_{alice,bob}.log`.
- **Idempotent:** re-runnable — uploads code, builds a `.venv-qne` (sequence 1.0.0) on both nodes, re-arms the switch loss, then runs.

## 1 · Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'                          # same slice as notebook 1
SCENARIO   = 'validation/scenarios/fabric_1km.yml'     # sizes the P4 loss model + run params

# BMv2 image: must match notebook 1 (container path). '' => switch built from source.
BMV2_IMAGE = 'ghcr.io/kthare10/qfabric-bmv2:latest'

# --- Channel mode (with or without the P4 switch) ---
TRANSPORT = 'raw'      # 'raw' (real 0x7101 frames) | 'tcp' (descriptors over TCP, no switch)
LOSS      = 'switch'   # 'switch' (BMv2 P4) | 'model' (software, no switch) | 'none' (lossless) | 'auto'

# Distributed-SeQUeNCe emulator knobs
NUM_PULSES      = 20000      # photons Alice emits
KEY_LENGTH      = 256        # target sifted key length (bits)
SAMPLE_FRACTION = 0.2        # fraction of sifted bits disclosed for QBER estimation
PHOTON_MODE     = 'bulk'     # 'bulk' (fast) or 'per_event' (per-photon fidelity) — see DESIGN §4.3
PHOTON_DRAIN_MS = 500        # wait for straggler photons after QUBITS_DONE (raw path vs TCP race)

## 2 · Load the slice

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

# deploy_fabric.py holds the tested provisioning/run logic; this notebook is a thin
# wrapper around it (single source of truth) — incl. the new SeQUeNCe-emulator helpers.
import deploy_fabric as df
from qne.config import ScenarioConfig

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## 3 · Ship the latest code + build the SeQUeNCe runtime

`upload_project` re-tars the repo to every node (this now includes `qne-sequence/`).
`setup_sequence_runtime` builds a dedicated `.venv-qne` (Python 3.12 + `sequence==1.0.0`
+ numpy) on **both** Alice and Bob and verifies the full import chain
(`sequence` + `qne` + `qne_sequence`). One-time per slice; idempotent (a few minutes).

In [ ]:
df.upload_project(slice_obj)
if BMV2_IMAGE:
    os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE   # configure_switch uses the container
df.setup_sequence_runtime(slice_obj)

## 4 · Arm the P4 loss model (only when using the switch)

When `TRANSPORT='raw'` and `LOSS` is `switch`/`auto`, the fiber loss is the **switch's**
job: `configure_switch` compiles/starts BMv2 and installs the per-wavelength loss
threshold from the scenario. For `tcp`, `loss=model`, or `loss=none` **no switch is
needed** and this step is skipped. (Data-plane IPs/ARP/promisc come from notebook 1.)

In [ ]:
cfg = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO)
USE_SWITCH = (TRANSPORT == 'raw' and LOSS in ('switch', 'auto'))
if USE_SWITCH:
    threshold = cfg.loss_threshold_u32
    print(f'Scenario {cfg.name}: distance={cfg.channel.distance_km} km, '
          f'atten={cfg.channel.attenuation_db_per_km} dB/km, '
          f'P(loss)={cfg.loss_probability:.4f}, P4 threshold(u32)={threshold}')
    df.configure_switch(slice_obj, threshold)   # (re)compile + start BMv2 + set loss table
else:
    print(f'No P4 switch needed (transport={TRANSPORT}, loss={LOSS}); skipping configure_switch.')

## 5 · Run distributed-SeQUeNCe BB84

Bob listens (TCP classical + photon RX); Alice connects over the data-plane IP and emits
photons per the chosen `TRANSPORT`/`LOSS`. Physics (fidelity/efficiency/dark counts) come
from the scenario; with `LOSS='none'` the channel is lossless (QBER from fidelity only).

In [ ]:
a_res, b_res = df.run_sequence_bb84(
    slice_obj,
    transport=TRANSPORT,
    loss=LOSS,
    num_pulses=NUM_PULSES,
    key_length=KEY_LENGTH,
    fidelity=cfg.channel.polarization_fidelity,
    efficiency=cfg.detector.efficiency,
    dark_count_rate=cfg.detector.dark_count_rate,
    distance_km=cfg.channel.distance_km,
    attenuation=cfg.channel.attenuation_db_per_km,
    sample_fraction=SAMPLE_FRACTION,
    photon_mode=PHOTON_MODE,
    photon_drain_ms=PHOTON_DRAIN_MS,
)

## 6 · Verify

In [ ]:
analytical = (1.0 - cfg.channel.polarization_fidelity) / 2.0
print(f'analytical intrinsic QBER (1-F)/2 = {analytical:.4f}\n')

checks = []
def rec(name, ok, detail=''):
    checks.append(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f' — {detail}' if detail else ''))

rec('alice produced a key', bool(a_res) and a_res.get('key') is not None)
rec('bob produced a key',   bool(b_res) and b_res.get('key') is not None)
if a_res:
    rec('alice: zero remote-access errors', a_res.get('remote_access_errors') == 0,
        f"errs={a_res.get('remote_access_errors')}")
    q = a_res.get('qber')
    rec('QBER plausible (< 0.11)', q is not None and 0 <= q < 0.11, f'QBER={q}')
    rec('sifted bits > key length', (a_res.get('sifted_bits') or 0) > KEY_LENGTH,
        f"sifted={a_res.get('sifted_bits')}")
    rec('secure fraction > 0', (a_res.get('secure_fraction') or 0) > 0,
        f"secure_fraction={a_res.get('secure_fraction')}")

print(f"\n(transport={a_res.get('quantum_transport') if a_res else '?'}, "
      f"loss_where={a_res.get('loss_where') if a_res else '?'})")
print('\nALL CHECKS PASSED' if checks and all(checks)
      else '\nSOME CHECKS FAILED — inspect /tmp/seq_{alice,bob}.log on the nodes'
           + (' and `sudo docker logs bmv2` (or journalctl -u bmv2) on the switch.' if USE_SWITCH else '.'))

## 7 · Notes & cleanup

- The slice stays up for re-runs. Sweep distance/fidelity by editing `SCENARIO` (or
  the knobs in cell 1) and re-running cells 4–6.
- **Tuning:** if Bob's `sifted_bits` looks short, raise `PHOTON_DRAIN_MS` (photons still
  in flight when `QUBITS_DONE` arrived over TCP), or lower `NUM_PULSES` if BMv2 drops
  under burst. Try `PHOTON_MODE='per_event'` for per-photon timing fidelity (slower).
- **Classical-network effects:** `df.apply_classical_netem(slice_obj, delay_ms=..., loss_pct=...)`
  impairs only TCP:5100 (photon plane untouched) — the QFabric research lever, here over
  the SeQUeNCe stack. Clear with `df.clear_classical_netem(slice_obj)`.
- **Delete the slice** when done: `df.cleanup(fablib, SLICE_NAME)`.